In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.daily_funnel AS
SELECT
  DATE(event_ts) AS event_date,
  device_type,
  COUNT_IF(event_type = 'page_view')   AS views,
  COUNT_IF(event_type = 'add_to_cart') AS carts,
  COUNT_IF(event_type = 'purchase')    AS purchases,
  ROUND(COUNT_IF(event_type = 'purchase') / NULLIF(COUNT_IF(event_type = 'page_view'), 0), 4) AS view_to_purchase_rate
FROM workspace.silver.events
GROUP BY DATE(event_ts), device_type;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.product_engagement AS
SELECT
  p.product_id, p.product_name, p.category,
  COUNT_IF(e.event_type = 'page_view')   AS views,
  COUNT_IF(e.event_type = 'add_to_cart') AS carts,
  COUNT_IF(e.event_type = 'purchase')    AS purchases
FROM workspace.silver.events e
JOIN workspace.silver.products p ON e.product_id = p.product_id
GROUP BY p.product_id, p.product_name, p.category;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.product_conversion AS
SELECT
  e.product_id, e.product_name, e.category,
  e.views, e.carts, e.purchases,
  ROUND(e.carts / NULLIF(e.views, 0), 4)     AS view_to_cart,
  ROUND(e.purchases / NULLIF(e.views, 0), 4) AS view_to_purchase,
  p.units_sold, p.revenue, p.return_rate
FROM workspace.gold.product_engagement e
JOIN workspace.gold.product_performance p ON e.product_id = p.product_id;

In [0]:
%sql
SELECT * FROM workspace.gold.product_engagement ORDER BY views DESC LIMIT 10;